# Week 0 · Part A: Train your first language model

**The LLM Resident** · the lab for *The Map of the Territory*

Thirty minutes, no GPU needed, no ML background assumed. You will train the smallest model that is honestly called a language model: it reads 32,033 human first names and learns which letter follows which. Then it invents new names, and you'll *measure* how good it is, because evaluation discipline starts on day one.

Run every cell top to bottom (`Shift+Enter`). The `assert` lines are your autograder: if a cell runs silently, you got it right.

## 1 · Get the data

32,033 names, one per line, collected from public records by Andrej Karpathy for his *makemore* series.

In [ ]:
import os, urllib.request

if not os.path.exists("names.txt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt",
        "names.txt",
    )
print("dataset ready:", os.path.getsize("names.txt"), "bytes")

## 2 · Look at it

Always look at your data before you model it. This habit will save you more times than any architecture trick in this course.

In [ ]:
words = open("names.txt").read().splitlines()

print("names:", len(words))
print("first five:", words[:5])
print("shortest:", min(words, key=len), "· longest:", max(words, key=len))

assert len(words) == 32033, "expected 32,033 names; re-run the download cell"
assert all(w.isalpha() and w.islower() for w in words), "unexpected characters in the data"

## 3 · Build a tokenizer (a tiny one)

Models eat numbers, not letters. Our vocabulary is 27 tokens: the letters `a–z`, plus `.` which marks both the **start** and the **end** of a name. Week 2 replaces this with a real BPE tokenizer you'll build from scratch; the idea is identical.

In [ ]:
chars = sorted(set("".join(words)))           # the 26 letters
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0                                 # '.' = start-of-name AND end-of-name
itos = {i: s for s, i in stoi.items()}

print("vocabulary size:", len(stoi))
assert len(stoi) == 27 and stoi["."] == 0 and itos[1] == "a"

## 4 · Train the model (yes, really)

The entire model is a 27×27 table: `N[a, b]` counts how often character `b` followed character `a` in the training data. "Training" is just counting. Every language model you have ever used is, at its core, a fancier way of filling in this table.

In [ ]:
import torch

N = torch.zeros((27, 27), dtype=torch.int32)   # [V, V]; row = current char, col = next char
for w in words:
    cs = ["."] + list(w) + ["."]
    for c1, c2 in zip(cs, cs[1:]):
        N[stoi[c1], stoi[c2]] += 1

assert N.shape == (27, 27)
assert int(N.sum()) == sum(len(w) + 1 for w in words), "every name contributes len+1 bigrams"

r, c = divmod(int(N.argmax()), 27)
print("total transitions counted:", int(N.sum()))
print(f"most common transition: {itos[r]!r} -> {itos[c]!r} ({int(N[r, c])} times)")

## 5 · Counts → probabilities

Each row becomes a probability distribution over the next character. The `+1` is *smoothing*: the model is never allowed to say "impossible," only "very unlikely." (Week 1 explains why an impossible-but-observed event would blow up our evaluation.)

In [ ]:
P = (N + 1).float()          # smoothing: no zero probabilities
P /= P.sum(1, keepdim=True)  # every row now sums to 1.0

assert torch.allclose(P.sum(1), torch.ones(27)), "rows must sum to 1"
print("P[stoi['q']]: what tends to follow 'q'? top guess:", itos[int(P[stoi['q']].argmax())])

## 6 · Generate

This loop (*look at the distribution for the current context, sample one token, append, repeat*) is the same loop ChatGPT runs. The only difference is how good the distribution is.

In [ ]:
g = torch.Generator().manual_seed(42)

for _ in range(10):
    out, ix = [], 0
    while True:
        ix = int(torch.multinomial(P[ix], num_samples=1, generator=g))
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))

## 7 · Measure it: evaluation starts today

Some of those names look plausible; some are garbage. "Looks OK to me" is not evaluation. The honest question: *how surprised is the model, on average, by real names?* That's the **average negative log-likelihood**: the single most important number in this entire course. Training a language model, at any scale, means making this number go down.

In [ ]:
log_likelihood, n = 0.0, 0
for w in words:
    cs = ["."] + list(w) + ["."]
    for c1, c2 in zip(cs, cs[1:]):
        log_likelihood += torch.log(P[stoi[c1], stoi[c2]])
        n += 1

nll = -log_likelihood / n
print(f"average negative log-likelihood: {float(nll):.4f}")
print("lower = less surprised = better. Write this number down;")
print("Week 1's neural model has to beat it, and you'll know exactly by how much.")

## What you just did

- **Trained a language model.** It looked at real data and now produces name-*shaped* strings it has never seen. That is not a metaphor for what GPT does; it *is* what GPT does, at 1/10-billionth scale.
- **It's terrible, measurably.** One character of context isn't enough, and you have the NLL number to prove it: not a vibe, a measurement.
- **You set up the course's core loop:** data → model → generate → *evaluate*. Every week deepens one of those arrows.

**Next:** Week 0 · Part B builds your toolchain and your shape-fluency; Week 1 replaces the counting table with a neural network, and beats your NLL number.

*The LLM Residency: six months, ten hours a week, no magic.*